<a href="https://colab.research.google.com/github/M7office/Stroke/blob/main/AHA_stress_repair_profile_trajectory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# ============================================================
# Slide 5 / Figure 5 — Time-dependent trajectories of Figure 1 composites
# Colab-ready script
#
# Input expected:
#   slide01_patient_level_profile_scores.csv
# The script searches common locations including the current folder
# and AHA_slide01_outputs_v*/ subfolders.
#
# Outputs:
#   AHA_slide05_outputs/
#   - ribbon trajectory figure
#   - point-range trajectory figure
# ============================================================

from pathlib import Path
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

# -----------------------------
# Settings
# -----------------------------
BASE = Path.cwd()
OUT = BASE / "AHA_slide05_outputs_v7"
OUT.mkdir(parents=True, exist_ok=True)

SIS3_CUTOFF = 63
RANDOM_SEED = 42
N_BOOT = 3000

TIME_ORDER = ["0–1y", "1–3y", ">3y"]
PROFILE_ORDER = [
    "Stress-response activation",
    "Repair / neurovascular remodeling",
    "Stress–repair imbalance",   # file name
]
PROFILE_DISPLAY = {
    "Stress-response activation": "Stress-response activation",
    "Repair / neurovascular remodeling": "Repair / neurovascular remodeling",
    "Stress–repair imbalance": "Stress-repair imbalance",
}

COLORS = {
    "lower": "#D55E00",
    "higher": "#0072B2",
    "black": "#303030",
    "gray": "#6E6E6E",
    "verylight": "#EAEAEA",
    "ribbon_lower": "#D55E00",
    "ribbon_higher": "#0072B2",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.8,
    "ytick.labelsize": 8.8,
    "legend.fontsize": 8.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 450,
})

# -----------------------------
# Helpers
# -----------------------------
def find_scores_file():
    candidates = [
        BASE / "slide01_patient_level_profile_scores.csv",
        BASE / "AHA_slide01_outputs_v3" / "slide01_patient_level_profile_scores.csv",
        BASE / "AHA_slide01_outputs_v2" / "slide01_patient_level_profile_scores.csv",
        BASE / "AHA_slide01_outputs" / "slide01_patient_level_profile_scores.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    # search recursively as fallback
    found = list(BASE.rglob("slide01_patient_level_profile_scores.csv"))
    if found:
        return found[0]
    raise FileNotFoundError("Could not find slide01_patient_level_profile_scores.csv")


def time_bin_from_years(y):
    if pd.isna(y):
        return np.nan
    if 0 <= y < 1:
        return "0–1y"
    if 1 <= y < 3:
        return "1–3y"
    if y >= 3:
        return ">3y"
    return np.nan


def mean_iqr(values):
    vals = pd.Series(values).dropna().astype(float).values
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    mean = float(np.mean(vals))
    q1, q3 = np.percentile(vals, [25, 75])
    return mean, float(q1), float(q3)


def round_down(x, base=0.1):
    return base * np.floor(x / base)

def round_up(x, base=0.1):
    return base * np.ceil(x / base)

def wrap_caption(text, width=185):
    return "\n".join(textwrap.wrap(text, width=width, break_long_words=False))


def smooth_curve(x, y, x_new):
    """
    Simple quadratic interpolation for 3 time bins.
    This is a visual smoothing aid only; original points are still shown.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) <= 1:
        return np.full_like(x_new, y[0] if len(y) else np.nan, dtype=float)
    deg = min(2, len(x) - 1)
    coeff = np.polyfit(x, y, deg=deg)
    return np.polyval(coeff, x_new)


# -----------------------------
# Load data
# -----------------------------
scores_file = find_scores_file()
df = pd.read_csv(scores_file)

needed = ["sis3", "time_years"] + PROFILE_ORDER
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in {scores_file.name}: {missing}")

df["time_bin"] = df["time_years"].map(time_bin_from_years)
df["sis3_group"] = np.where(df["sis3"] <= SIS3_CUTOFF, "Lower SIS3 (≤63)", "Higher SIS3 (>63)")
df = df.dropna(subset=["time_bin"]).copy()
df["time_bin"] = pd.Categorical(df["time_bin"], categories=TIME_ORDER, ordered=True)

# -----------------------------
# Summaries
# -----------------------------
rows = []
seed_counter = 0
for profile in PROFILE_ORDER:
    for sis3_group in ["Lower SIS3 (≤63)", "Higher SIS3 (>63)"]:
        for time_bin in TIME_ORDER:
            vals = pd.to_numeric(
                df.loc[(df["sis3_group"] == sis3_group) & (df["time_bin"] == time_bin), profile],
                errors="coerce"
            ).dropna()
            mean, lo, hi = mean_iqr(vals)
            seed_counter += 1
            rows.append({
                "profile": profile,
                "profile_display": PROFILE_DISPLAY[profile],
                "sis3_group": sis3_group,
                "time_bin": time_bin,
                "n": int(len(vals)),
                "mean": mean,
                "ci_low": lo,
                "ci_high": hi,
            })

summary = pd.DataFrame(rows)
summary.to_csv(OUT / "slide05_profile_trajectory_summary_iqr.csv", index=False)

# Global y-range across all panels
ymin = round_down(np.nanmin(summary["ci_low"]) - 0.04, 0.1)
ymax = round_up(np.nanmax(summary["ci_high"]) + 0.04, 0.1)

# Keep a reasonable, consistent range
ymin = min(ymin, -0.4)
ymax = max(ymax, 0.6)

# -----------------------------
# Shared drawing function
# -----------------------------
def make_figure(mode="ribbon"):
    if mode == "ribbon":
        fig_title = "Figure 5. Stress-Repair Profile Values Show Time-Dependent Trajectories"
        out_stem = "figure5_profile_trajectories_ribbon_iqr_journal_style_v7"
    else:
        fig_title = "Figure 5B. Stress-Repair Profile Values Show Time-Dependent Trajectories"
        out_stem = "figure5_composite_trajectories_pointrange_journal_style_v4"

    fig = plt.figure(figsize=(14.8, 8.15))
    gs = GridSpec(
        4, 3,
        figure=fig,
        height_ratios=[0.42, 4.45, 0.58, 1.20],
        hspace=0.18,
        wspace=0.26,
    )

    title_ax = fig.add_subplot(gs[0, :])
    title_ax.axis("off")
    title_ax.text(
        0.00, 0.72, fig_title,
        ha="left", va="center", fontsize=13.0, fontweight="bold",
        color=COLORS["black"], transform=title_ax.transAxes
    )

    axes = []
    x = np.arange(len(TIME_ORDER))
    for j, profile in enumerate(PROFILE_ORDER):
        ax = fig.add_subplot(gs[1, j], sharey=axes[0] if axes else None)
        axes.append(ax)

        for group, color in [("Lower SIS3 (≤63)", COLORS["lower"]), ("Higher SIS3 (>63)", COLORS["higher"])]:
            sub = summary[(summary["profile"] == profile) & (summary["sis3_group"] == group)].copy()
            sub["x"] = sub["time_bin"].map({t: i for i, t in enumerate(TIME_ORDER)})

            if mode == "ribbon":
                means = sub["mean"].values.astype(float)
                lo = sub["ci_low"].values.astype(float)
                hi = sub["ci_high"].values.astype(float)
                x_smooth = np.linspace(x.min(), x.max(), 200)
                mean_smooth = smooth_curve(x, means, x_smooth)
                lo_smooth = smooth_curve(x, lo, x_smooth)
                hi_smooth = smooth_curve(x, hi, x_smooth)

                ax.fill_between(
                    x_smooth,
                    lo_smooth,
                    hi_smooth,
                    color=color, alpha=0.16, linewidth=0
                )
                ax.plot(x_smooth, mean_smooth, color=color, lw=1.7)
                ax.plot(x, means, color=color, marker="o", markersize=4.8, lw=0)
            else:
                means = sub["mean"].values.astype(float)
                lo = sub["ci_low"].values.astype(float)
                hi = sub["ci_high"].values.astype(float)
                yerr = np.vstack([means - lo, hi - means])
                ax.errorbar(
                    x, means, yerr=yerr,
                    color=color, marker="o", markersize=5, lw=1.6,
                    elinewidth=1.2, capsize=3, capthick=1.0
                )
                ax.plot(x, means, color=color, lw=1.3)

        ax.set_title(PROFILE_DISPLAY[profile], pad=8, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(TIME_ORDER)
        ax.set_xlabel("Time since stroke")
        ax.set_ylim(ymin, ymax)
        ax.grid(axis="y", color=COLORS["verylight"], lw=0.7)
        ax.set_axisbelow(True)
        ax.tick_params(axis="both", length=3, color=COLORS["gray"])
        ax.set_ylabel("Mean standardized profile value")
        ax.tick_params(axis="y", labelleft=True)

    legend_ax = fig.add_subplot(gs[2, :])
    legend_ax.axis("off")
    handles = [
        Line2D([0], [0], color=COLORS["lower"], marker="o", lw=1.6, markersize=5, label="Lower SIS3 (≤63)"),
        Line2D([0], [0], color=COLORS["higher"], marker="o", lw=1.6, markersize=5, label="Higher SIS3 (>63)"),
    ]
    legend_ax.legend(
        handles=handles,
        ncol=2,
        frameon=False,
        loc="center",
        bbox_to_anchor=(0.5, 0.35),
        handlelength=2.0,
        columnspacing=2.0
    )

    cap_ax = fig.add_subplot(gs[3, :])
    cap_ax.axis("off")
    if mode == "ribbon":
        caption = (
            "Profile trajectories are shown for the 3 hypothesis-guided profiles highlighted in Figure 1: stress-response activation, "
            "repair / neurovascular remodeling, and stress-repair imbalance. Patients were grouped as lower SIS3 (≤63) or higher SIS3 (>63), "
            "and time since stroke was grouped as 0–1 year, 1–3 years, and more than 3 years. Points indicate mean standardized profile values; shaded bands "
            "indicate the interquartile range (25th–75th percentile). All panels use the same y-axis range to facilitate direct visual comparison across profiles."
        )
    else:
        caption = (
            "Profile trajectories are shown for the 3 hypothesis-guided profiles highlighted in Figure 1: stress-response activation, "
            "repair / neurovascular remodeling, and stress-repair imbalance. Patients were grouped as lower SIS3 (≤63) or higher SIS3 (>63), "
            "and time since stroke was grouped as 0–1 year, 1–3 years, and more than 3 years. Points indicate mean composite scores, vertical bars "
            "indicate bootstrap 95% CIs, and lines connect means across time bins. All panels use the same y-axis range to facilitate direct visual comparison across profiles."
        )
    cap_ax.text(
        0.00, 0.64,
        wrap_caption(caption, width=190),
        ha="left", va="top", fontsize=8.5, color=COLORS["black"],
        linespacing=1.18, transform=cap_ax.transAxes
    )

    fig.subplots_adjust(left=0.07, right=0.98, top=0.93, bottom=0.09)
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(OUT / f"{out_stem}.{ext}", bbox_inches="tight")
    plt.close(fig)


make_figure(mode="ribbon")

print("Created Slide 5 outputs:")
for p in sorted(OUT.glob("figure5_*")):
    print(" -", p.name)


Created Slide 5 outputs:
 - figure5_profile_trajectories_ribbon_iqr_journal_style_v7.pdf
 - figure5_profile_trajectories_ribbon_iqr_journal_style_v7.png
 - figure5_profile_trajectories_ribbon_iqr_journal_style_v7.svg
